In [ ]:
%%sql

DROP TABLE IF EXISTS silver_rdm_session_status_add;

CREATE TABLE silver_rdm_session_status_add AS

WITH mpb_source AS (
    -- MPB source values derived from DRJ appointments and appointment attendances
    -- session_status_src_name uses attendance name + cancelledBy for MPB001

    SELECT DISTINCT
        CONCAT(COALESCE(TRIM(att.name), 'Unknown'), '_', COALESCE(TRIM(a.cancelledBy), 'Unknown')) AS session_status_src_name,
        LOWER(TRIM(CONCAT(COALESCE(TRIM(att.name), 'Unknown'), '_', COALESCE(TRIM(a.cancelledBy), 'Unknown')))) AS session_status_src_id,
        'MPB001' AS session_status_src_sys_inst_id
    FROM silver_drj_appointments a
    LEFT JOIN silver_drj_appointment_attendances att
        ON a.attendance_id = att.id
    WHERE (a.attendance_id IS NOT NULL OR a.cancelledBy IS NOT NULL)
      AND TRIM(CONCAT(COALESCE(TRIM(att.name), 'Unknown'), '_', COALESCE(TRIM(a.cancelledBy), 'Unknown'))) <> ''
)

SELECT
    s.session_status_src_id,
    s.session_status_src_name,
    s.session_status_src_sys_inst_id
FROM mpb_source s
LEFT JOIN silver_rdm_session_status r
    ON LOWER(TRIM(s.session_status_src_id)) = LOWER(TRIM(r.session_status_src_id))
WHERE r.session_status_src_id IS NULL;

In [ ]:
%%sql
SELECT *
FROM silver_rdm_session_status_add
ORDER BY session_status_src_name;

In [ ]:
%%sql
SELECT COUNT(*) AS total_rows
FROM silver_rdm_session_status_add;

In [ ]:
%%sql
SELECT
    session_status_src_id,
    session_status_src_sys_inst_id,
    COUNT(*) AS cnt
FROM silver_rdm_session_status_add
GROUP BY
    session_status_src_id,
    session_status_src_sys_inst_id
HAVING COUNT(*) > 1;

In [ ]:
%%sql
SELECT
    a.session_status_src_id,
    a.session_status_src_name,
    a.session_status_src_sys_inst_id,
    r.session_status_src_id AS existing_rdm_id
FROM silver_rdm_session_status_add a
LEFT JOIN silver_rdm_session_status r
    ON LOWER(TRIM(a.session_status_src_id)) = LOWER(TRIM(r.session_status_src_id))
   AND LOWER(TRIM(a.session_status_src_sys_inst_id)) = LOWER(TRIM(r.session_status_src_sys_inst_id))
ORDER BY a.session_status_src_name;

In [ ]:
%%sql
SELECT *
FROM silver_rdm_session_status_add
WHERE session_status_src_sys_inst_id = 'MPB001'
ORDER BY session_status_src_name;

In [ ]:
MPB session_status add-table logic completed in silver_rdm_session_status_add.

Source used:
- silver_drj_appointments
- silver_drj_appointment_attendances

Derived fields created:
- session_status_src_id
- session_status_src_name
- session_status_src_sys_inst_id = MPB001

Validation:
- total rows loaded: 55
- duplicate check: no duplicate rows returned

Next step:
- append WIP logic into the same silver_rdm_session_status_add table
- then append S1 logic

In [ ]:
%%sql

DROP TABLE IF EXISTS silver_rdm_session_status_add;

CREATE TABLE silver_rdm_session_status_add AS

WITH mpb_source AS (
    -- MPB source values derived from DRJ appointments and appointment attendances
    -- MPB session_status_src_id uses the same logic as session_status_src_name for MPB001

    SELECT DISTINCT
        CONCAT(COALESCE(TRIM(att.name), 'Unknown'), '_', COALESCE(TRIM(a.cancelledBy), 'Unknown')) AS session_status_src_name,
        LOWER(TRIM(CONCAT(COALESCE(TRIM(att.name), 'Unknown'), '_', COALESCE(TRIM(a.cancelledBy), 'Unknown')))) AS session_status_src_id,
        'MPB001' AS session_status_src_sys_inst_id
    FROM silver_drj_appointments a
    LEFT JOIN silver_drj_appointment_attendances att
        ON a.attendance_id = att.id
    WHERE (a.attendance_id IS NOT NULL OR a.cancelledBy IS NOT NULL)
      AND TRIM(CONCAT(COALESCE(TRIM(att.name), 'Unknown'), '_', COALESCE(TRIM(a.cancelledBy), 'Unknown'))) <> ''
)

SELECT
    s.session_status_src_id,
    s.session_status_src_name,
    s.session_status_src_sys_inst_id
FROM mpb_source s
LEFT JOIN silver_rdm_session_status r
    ON LOWER(TRIM(s.session_status_src_id)) = LOWER(TRIM(r.session_status_src_id))
WHERE r.session_status_src_id IS NULL;